# 01 — Encoder / 数据编码

**Chapter 22 — File 4 of 8 / 第22章 — 第4个文件（共8个）**

---

## Summary / 总结

This script demonstrates **Implementing the Add & Norm Layer**.

本脚本演示 **Implementing the Add & Norm Layer**。

---
## Step 1 — Step 1

In [ ]:
from tensorflow.keras.layers import LayerNormalization, Layer, Dense, ReLU, Dropout
from multihead_attention import MultiHeadAttention
from positional_encoding import PositionEmbeddingFixedWeights

---
## Step 2 — Implementing the Add & Norm Layer

In [ ]:
class AddNormalization(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.layer_norm = LayerNormalization()  # Layer normalization layer

    def call(self, x, sublayer_x):

---
## Step 3 — The sublayer input and output need to be of the same shape to be summed

In [ ]:
add = x + sublayer_x

---
## Step 4 — Apply layer normalization to the sum

In [ ]:
return self.layer_norm(add)

---
## Step 5 — Implementing the Feed-Forward Layer

In [ ]:
class FeedForward(Layer):
    def __init__(self, d_ff, d_model, **kwargs):
        super().__init__(**kwargs)
        self.fully_connected1 = Dense(d_ff)  # First fully connected layer
        self.fully_connected2 = Dense(d_model)  # Second fully connected layer
        self.activation = ReLU()  # ReLU activation layer

    def call(self, x):

---
## Step 6 — The input is passed into the two fully-connected layers, with a ReLU in between

In [ ]:
x_fc1 = self.fully_connected1(x)

        return self.fully_connected2(self.activation(x_fc1))

---
## Step 7 — Implementing the Encoder Layer

In [ ]:
class EncoderLayer(Layer):
    def __init__(self, h, d_k, d_v, d_model, d_ff, rate, **kwargs):
        super().__init__(**kwargs)
        self.multihead_attention = MultiHeadAttention(h, d_k, d_v, d_model)
        self.dropout1 = Dropout(rate)
        self.add_norm1 = AddNormalization()
        self.feed_forward = FeedForward(d_ff, d_model)
        self.dropout2 = Dropout(rate)
        self.add_norm2 = AddNormalization()

    def call(self, x, padding_mask, training):

---
## Step 8 — Multi-head attention layer

In [ ]:
multihead_output = self.multihead_attention(x, x, x, padding_mask)

---
## Step 9 — Expected output shape = (batch_size, sequence_length, d_model)
Add in a dropout layer

In [ ]:
multihead_output = self.dropout1(multihead_output, training=training)

---
## Step 10 — Followed by an Add & Norm layer

In [ ]:
addnorm_output = self.add_norm1(x, multihead_output)

---
## Step 11 — Expected output shape = (batch_size, sequence_length, d_model)
Followed by a fully connected layer

In [ ]:
feedforward_output = self.feed_forward(addnorm_output)

---
## Step 12 — Expected output shape = (batch_size, sequence_length, d_model)
Add in another dropout layer

In [ ]:
feedforward_output = self.dropout2(feedforward_output, training=training)

---
## Step 13 — Followed by another Add & Norm layer

In [ ]:
return self.add_norm2(addnorm_output, feedforward_output)

---
## Step 14 — Implementing the Encoder

In [ ]:
class Encoder(Layer):
    def __init__(self, vocab_size, sequence_length, h, d_k, d_v, d_model, d_ff, n, rate,
                       **kwargs):
        super().__init__(**kwargs)
        self.pos_encoding = PositionEmbeddingFixedWeights(sequence_length, vocab_size,
                                                          d_model)
        self.dropout = Dropout(rate)
        self.encoder_layer = [EncoderLayer(h, d_k, d_v, d_model, d_ff, rate)
                              for _ in range(n)]

    def call(self, input_sentence, padding_mask, training):

---
## Step 15 — Generate the positional encoding

In [ ]:
pos_encoding_output = self.pos_encoding(input_sentence)

---
## Step 16 — Expected output shape = (batch_size, sequence_length, d_model)
Add in a dropout layer

In [ ]:
x = self.dropout(pos_encoding_output, training=training)

---
## Step 17 — Pass on the positional encoded values to each encoder layer

In [ ]:
for i, layer in enumerate(self.encoder_layer):
            x = layer(x, padding_mask, training)

        return x

---
## Learning Notes / 学习笔记

- **概念**: Implementing the Add & Norm Layer 是机器学习中的常用技术。  
  *Implementing the Add & Norm Layer is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Encoder / 数据编码
# Complete Code / 完整代码
# ===============================

from tensorflow.keras.layers import LayerNormalization, Layer, Dense, ReLU, Dropout
from multihead_attention import MultiHeadAttention
from positional_encoding import PositionEmbeddingFixedWeights

# Implementing the Add & Norm Layer
class AddNormalization(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.layer_norm = LayerNormalization()  # Layer normalization layer

    def call(self, x, sublayer_x):
        # The sublayer input and output need to be of the same shape to be summed
        add = x + sublayer_x

        # Apply layer normalization to the sum
        return self.layer_norm(add)

# Implementing the Feed-Forward Layer
class FeedForward(Layer):
    def __init__(self, d_ff, d_model, **kwargs):
        super().__init__(**kwargs)
        self.fully_connected1 = Dense(d_ff)  # First fully connected layer
        self.fully_connected2 = Dense(d_model)  # Second fully connected layer
        self.activation = ReLU()  # ReLU activation layer

    def call(self, x):
        # The input is passed into the two fully-connected layers, with a ReLU in between
        x_fc1 = self.fully_connected1(x)

        return self.fully_connected2(self.activation(x_fc1))

# Implementing the Encoder Layer
class EncoderLayer(Layer):
    def __init__(self, h, d_k, d_v, d_model, d_ff, rate, **kwargs):
        super().__init__(**kwargs)
        self.multihead_attention = MultiHeadAttention(h, d_k, d_v, d_model)
        self.dropout1 = Dropout(rate)
        self.add_norm1 = AddNormalization()
        self.feed_forward = FeedForward(d_ff, d_model)
        self.dropout2 = Dropout(rate)
        self.add_norm2 = AddNormalization()

    def call(self, x, padding_mask, training):
        # Multi-head attention layer
        multihead_output = self.multihead_attention(x, x, x, padding_mask)
        # Expected output shape = (batch_size, sequence_length, d_model)

        # Add in a dropout layer
        multihead_output = self.dropout1(multihead_output, training=training)

        # Followed by an Add & Norm layer
        addnorm_output = self.add_norm1(x, multihead_output)
        # Expected output shape = (batch_size, sequence_length, d_model)

        # Followed by a fully connected layer
        feedforward_output = self.feed_forward(addnorm_output)
        # Expected output shape = (batch_size, sequence_length, d_model)

        # Add in another dropout layer
        feedforward_output = self.dropout2(feedforward_output, training=training)

        # Followed by another Add & Norm layer
        return self.add_norm2(addnorm_output, feedforward_output)

# Implementing the Encoder
class Encoder(Layer):
    def __init__(self, vocab_size, sequence_length, h, d_k, d_v, d_model, d_ff, n, rate,
                       **kwargs):
        super().__init__(**kwargs)
        self.pos_encoding = PositionEmbeddingFixedWeights(sequence_length, vocab_size,
                                                          d_model)
        self.dropout = Dropout(rate)
        self.encoder_layer = [EncoderLayer(h, d_k, d_v, d_model, d_ff, rate)
                              for _ in range(n)]

    def call(self, input_sentence, padding_mask, training):
        # Generate the positional encoding
        pos_encoding_output = self.pos_encoding(input_sentence)
        # Expected output shape = (batch_size, sequence_length, d_model)

        # Add in a dropout layer
        x = self.dropout(pos_encoding_output, training=training)

        # Pass on the positional encoded values to each encoder layer
        for i, layer in enumerate(self.encoder_layer):
            x = layer(x, padding_mask, training)

        return x

---

➡️ **Next / 下一步**: File 5 of 8